# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [ ]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | model.joblib | Separa el objeto ejecutable de los metadatos: solo se deserializa cuando el manifiesto ya ha dicho que es compatible |
| Versión del esquema | manifest.json (schema_version) | Que el cargador lea un manifiesto con una estructura que no conoce |
| Orden y nombres de features | manifest.json (feature_names), comparado con FEATURE_NAMES del código | Que el modelo reciba columnas cambiadas de sitio sin avisar (error silencioso) |
| Versión del preprocesado | manifest.json (preprocessing_version), comparado con PREPROCESSING_VERSION del código | Usar el modelo con una transformación distinta a la del entrenamiento |
| Etiquetas de salida permitidas | manifest.json (output_labels) y el contrato de salida en código | Que la respuesta exponga categorías que la interfaz no entiende |
| Versión del modelo | manifest.json (model_version) y en cada predicción | No saber qué modelo produjo una predicción ni poder volver a una versión anterior |
| Tipo de estimador | manifest.json (estimator_type) | Saber qué clase de modelo hay dentro sin abrir el binario y detectar un objeto inesperado |

Las reglas (rangos de entrada, comprobación de la interfaz predict/predict_proba, política de errores) viven en el código y en los tests, no en el manifiesto.

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**: la carga se detiene con un error que dice que el bundle debe contener manifest.json y model.joblib. No se carga el modelo.
2. aparecen features desconocidas o en un orden distinto, la validación del manifiesto falla porque feature_names no coincide con FEATURE_NAMES. No se carga el modelo.
3. el modelo devuelve una etiqueta que no está en **output_labels** el contrato de salida (WineQualityPrediction) rechaza la predicción. No se escribe el CSV.
4. una fila del CSV no cumple el contrato de entrada: WineQualityRequest la rechaza, el CLI muestra el error con el sample_id y termina con código 2. No se escribe el CSV.

In [ ]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "wine-quality-bundle-v1",
    "model_version": legacy_model_version,
    "preprocessing_version": "wine-red-features-v1",
    "feature_names": feature_names,
    "output_labels": ["needs_review", "acceptable", "excellent"],
    "estimator_type": "RandomForestClassifier",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.


| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian density y alcohol | Carga del bundle: validación del manifiesto (feature_names frente a FEATURE_NAMES) | feature_names no coincide con el orden del contrato | No |
| Cambia preprocessing_version | Carga del bundle: validación del manifiesto | preprocessing_version incompatible: se esperaba wine-red-features-v1 | No |
| El modelo devuelve unknown | Inferencia: contrato de salida (WineQualityPrediction) | quality_band debe ser needs_review, acceptable o excellent | No |
| La petición incluye ph = 99 | Contrato de entrada (WineQualityRequest) al leer el CSV | Muestra red-001: ph debe ser menor o igual que 4.5 | No |
| El estimador no tiene predict_proba (otro tipo de objeto) | Carga del bundle: comprobación de la interfaz del estimador | El estimador debe exponer predict y predict_proba | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
  Las seis claves son obligatorias. Ninguna tiene valor por defecto.

- ¿Aceptarás campos desconocidos en el manifiesto?
  No, se rechazan con extra="forbid".

- ¿Validarás el manifiesto antes de cargar el objeto serializado?
  Sí. Primero manifest.json y, si es compatible, después model.joblib.

- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
  Solo al final, cuando todas las filas son válidas.

- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
  Un error por stderr con la muestra y el campo que falla, y código de salida 2.

- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?
  Se para todo y no se escribe ningún CSV, para no publicar una salida incompleta.

### Invariantes que bloquean la carga o la inferencia

1. schema_version y preprocessing_version coinciden con los del código.
2. feature_names tiene los 11 nombres en el orden de FEATURE_NAMES.
3. El estimador tiene predict y predict_proba.
4. La predicción tiene una quality_band válida y una confianza entre 0 y 1.

La segunda clase convertirá estas decisiones en ArtifactManifest, save_model_bundle, load_model_bundle, inferencia y pruebas.